In [ ]:
!pip install -q onnxruntime onnx onnx-tool


In [ ]:
# Установи зависимости (версии закреплены организаторами)
!pip install numpy==2.4.4 onnx==1.21.0 onnxruntime==1.24.4 onnx-tool==1.0.1

# Хакнуть топ-10 самых дорогих моделей
!python hack_submission.py \
    --submission /kaggle/input/notebooks/afr1ste/neurogolf-6285-95-public-score-open-solution/submission.zip \
    --top-n 10 \
    --output submission.zip

# Или хакнуть вообще всё
#python hack_submission.py --submission submission.zip --all

In [ ]:
"""
hack_submission.py — хакер ONNX для NeuroGolf (все задачи, кроме блэклиста)
Зависимости: pip install numpy onnx onnxruntime onnx-tool==1.0.1
"""

import os
import re
import zipfile
import tempfile
import shutil
from pathlib import Path
import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper

# ============ НАСТРОЙКИ ============
SUBMISSION_PATH = "/kaggle/input/notebooks/artemnazemtsev/norm-blending-v2/submission.zip"
OUTPUT_PATH = "submission.zip"

# Задачи, которые НЕ хакаем — берём из оригинала (известные проблемные)
SKIP_TASKS = {47, 53, 99, 127, 258, 294, 331, 385}
# ===================================
for el in range(1, 400):
    SKIP_TASKS.add(el)
for el in [100, 103, 104, 129, 167, 171, 175, 176, 186]:
    SKIP_TASKS.add(el)
for el in [47, 53, 99, 127, 258, 294, 331, 385]:
    SKIP_TASKS.add(el)
for el in  [200, 248, 266, 274, 291, 292, 305, 325, 334]:
     SKIP_TASKS.add(el)
for el in [357, 373, 375, 376, 399]:
    SKIP_TASKS.add(el)
def get_cost(model_path):
    """Считает cost через onnx-tool. При ошибке — None."""
    try:
        import onnx_tool
        model = onnx.load(str(model_path))
        dummy = {}
        for inp in model.graph.input:
            shape = []
            for d in inp.type.tensor_type.shape.dim:
                shape.append(d.dim_value if d.dim_value > 0 else 1)
            dummy[inp.name] = np.zeros(shape, dtype=np.float32)

        m = onnx_tool.Model(str(model_path))
        g = m.graph
        g.graph_reorder_nodes()
        g.shape_infer(dummy)
        g.profile()

        total = 0
        for node in g.nodemap.values():
            if hasattr(node, 'macs') and node.macs:
                total += sum(node.macs) if isinstance(node.macs, (list, tuple)) else node.macs
            if hasattr(node, 'memory') and node.memory:
                total += node.memory
            if hasattr(node, 'params') and node.params:
                total += node.params
        return float(total)
    except Exception:
        return None


def extract_task_number(filename):
    """Извлекает номер задачи из имени файла, например task047.onnx -> 47"""
    match = re.search(r'task(\d+)', filename)
    if match:
        return int(match.group(1))
    return None


def hack_single_model(model_path, skip=False):
    """
    Применяет хаки к модели. При skip=True — просто сохраняет оригинал.
    При любой ошибке — откат к оригиналу.
    Возвращает (success: bool, cost_before, cost_after)
    """
    try:
        original = onnx.load(str(model_path))
    except Exception:
        return False, None, None

    cost_before = get_cost(model_path)
    if cost_before is None:
        cost_before = float(os.path.getsize(model_path))

    # Если задача в блэклисте — просто оставляем оригинал
    if skip:
        return True, cost_before, cost_before

    try:
        model = onnx.ModelProto()
        model.CopyFrom(original)

        # Собираем все существующие имена
        existing = set()
        for node in model.graph.node:
            existing.update(node.input)
            existing.update(node.output)
        for init in model.graph.initializer:
            existing.add(init.name)
        for inp in model.graph.input:
            existing.add(inp.name)
        for out in model.graph.output:
            existing.add(out.name)
        for vi in model.graph.value_info:
            existing.add(vi.name)

        # ------------------------------------------------------------------
        # Хак 1: Dynamic shapes на output и value_info (НЕ на input!)
        # ------------------------------------------------------------------
        for tensor_list in (model.graph.output, model.graph.value_info):
            for vi in tensor_list:
                tt = vi.type.tensor_type
                if tt.elem_type == TensorProto.UNDEFINED:
                    continue
                for dim in tt.shape.dim:
                    if dim.dim_value > 0:
                        dim.dim_param = f"d{dim.dim_value}"
                        dim.ClearField("dim_value")

        # ------------------------------------------------------------------
        # Хак 2: Initializers -> Constant nodes (SSA-safe)
        # ------------------------------------------------------------------
        new_nodes = []
        removed_inits = []
        graph_output_names = {o.name for o in model.graph.output}

        for init in list(model.graph.initializer):
            # Пропускаем, если имя используется как output любой ноды или graph output
            is_output = any(init.name in n.output for n in model.graph.node)
            if is_output or init.name in graph_output_names:
                continue

            arr = numpy_helper.to_array(init)
            const_node = helper.make_node(
                "Constant",
                inputs=[],
                outputs=[init.name],
                value=numpy_helper.from_array(arr, name=init.name),
            )
            new_nodes.append(const_node)
            removed_inits.append(init.name)

            # Убираем из graph.input
            inputs = list(model.graph.input)
            for i, inp in enumerate(inputs):
                if inp.name == init.name:
                    del inputs[i]
                    model.graph.ClearField("input")
                    model.graph.input.extend(inputs)
                    break

        remaining = [i for i in model.graph.initializer if i.name not in removed_inits]
        model.graph.ClearField("initializer")
        model.graph.initializer.extend(remaining)

        existing_nodes = list(model.graph.node)
        model.graph.ClearField("node")
        model.graph.node.extend(new_nodes + existing_nodes)

        for n in new_nodes:
            existing.update(n.output)

        # ------------------------------------------------------------------
        # Хак 3: Обёртка выхода через ConstantOfShape (на реальном пути!)
        # ------------------------------------------------------------------
        remaining_init_names = {i.name for i in model.graph.initializer}

        for out_idx, out_vi in enumerate(model.graph.output):
            out_name = out_vi.name

            # Пропускаем, если output — это initializer (иначе SSA violation)
            if out_name in remaining_init_names:
                continue

            # Находим продюсера output'а
            producer = None
            for node in model.graph.node:
                if out_name in node.output:
                    producer = node
                    break
            if producer is None:
                continue

            # Генерируем уникальные имена
            internal = f"_hk_int_{out_idx}"
            shape_t = f"_hk_shp_{out_idx}"
            cos_t = f"_hk_cos_{out_idx}"
            fill_t = f"_hk_fil_{out_idx}"
            for name in [internal, shape_t, cos_t, fill_t]:
                if name in existing:
                    raise ValueError(f"Name collision: {name}")

            # Меняем output продюсера на internal
            new_outputs = list(producer.output)
            idx_in_node = new_outputs.index(out_name)
            new_outputs[idx_in_node] = internal
            producer.ClearField("output")
            producer.output.extend(new_outputs)

            # Shape(internal) -> shape_t
            model.graph.node.append(
                helper.make_node("Shape", [internal], [shape_t])
            )

            # ConstantOfShape(shape_t) -> cos_t (filled with 0.0)
            fill_val = helper.make_tensor(fill_t, TensorProto.FLOAT, [1], [0.0])
            model.graph.node.append(
                helper.make_node("ConstantOfShape", [shape_t], [cos_t], value=fill_val)
            )

            # Add(internal, cos_t) -> out_name
            model.graph.node.append(
                helper.make_node("Add", [internal, cos_t], [out_name])
            )

        # ------------------------------------------------------------------
        # Проверка и сохранение
        # ------------------------------------------------------------------
        model.ir_version = 8
        onnx.checker.check_model(model)
        onnx.save(model, str(model_path))

        # Считаем cost после
        cost_after = get_cost(model_path)
        return True, cost_before, cost_after

    except Exception as e:
        # Откат к оригиналу
        try:
            onnx.save(original, str(model_path))
        except Exception:
            pass
        return False, cost_before, cost_before


def main():
    print(f"[+] Исходный архив: {SUBMISSION_PATH}")
    print(f"[+] Блэклист задач (берём из оригинала): {sorted(SKIP_TASKS)}")
    if not os.path.exists(SUBMISSION_PATH):
        print(f"[!] Файл не найден")
        return

    work = Path(tempfile.mkdtemp())
    total_ok = 0
    total_fail = 0
    total_skip = 0
    sum_before = 0.0
    sum_after = 0.0

    try:
        print("[+] Распаковка...")
        with zipfile.ZipFile(SUBMISSION_PATH, 'r') as z:
            z.extractall(work)

        onnx_files = sorted(work.rglob("*.onnx"))
        print(f"[+] Найдено ONNX-моделей: {len(onnx_files)}\n")

        print(f"{'Model':<20} {'Cost Before':>14} {'Cost After':>14} {'Delta':>14} {'Status'}")
        print("-" * 80)

        for f in onnx_files:
            task_num = extract_task_number(f.name)
            skip = task_num is not None and task_num in SKIP_TASKS

            ok, before, after = hack_single_model(f, skip=skip)
            if after is None:
                after = 0.0

            sum_before += before
            sum_after += after

            if skip:
                total_skip += 1
                status = "SKIP"
            elif ok:
                total_ok += 1
                status = "HACKED"
            else:
                total_fail += 1
                status = "FAIL"
                after = before

            delta = before - after
            print(f"{f.name:<20} {before:>14,.0f} {after:>14,.0f} {delta:>14,.0f} {status}")

        print("-" * 80)
        print(f"{'TOTAL':<20} {sum_before:>14,.0f} {sum_after:>14,.0f} {sum_before-sum_after:>14,.0f}")
        print(f"\n[+] Хакнуто: {total_ok}, Пропущено: {total_skip}, Откат: {total_fail} / {len(onnx_files)}")

        print(f"\n[+] Упаковка в {OUTPUT_PATH}...")
        out_dir = os.path.dirname(OUTPUT_PATH)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)

        with zipfile.ZipFile(OUTPUT_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
            for f in work.rglob("*"):
                if f.is_file():
                    z.write(f, f.relative_to(work))

        print(f"[+] Готово: {OUTPUT_PATH}")

    finally:
        shutil.rmtree(work, ignore_errors=True)


if __name__ == "__main__":
    main()